# GRAST — WP07.3 TinyStories pretraining

Chọn GPU A100 trong Runtime → Change runtime type. Notebook lưu mọi run/checkpoint trực tiếp vào Google Drive và luôn in log từng dòng. Mặc định chỉ smoke 4 updates; sau khi kiểm tra thành công, đặt `SMOKE_ONLY = False` để chạy budget 2.000 updates. Đây là recipe khởi đầu, chưa phải kết quả baseline.

Chuẩn bị file `grast-wp07-data-block256-17971c0168e4.zip` (khoảng 3,18 MiB) từ thư mục dist của project. Bundle block64 ở WP07.2 không dùng cho recipe block256 này. Train/validation giữ nguyên split WP03; không có test set trong bundle.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("GRAST requires Python 3.11–3.13")
os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
REPO = Path("/content/grast-wp07")
REVISION = "feat/wp07-3-pretraining-runner"  # replaced with verified commit before delivery


def run_logged(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        bufsize=1,
        env=os.environ.copy(),
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            status = process.wait()
        except BaseException:
            import signal

            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
            raise
        if status:
            raise subprocess.CalledProcessError(status, command)


if not REPO.exists():
    run_logged(["git", "clone", "https://github.com/CuongBien/GRAST.git", REPO])
run_logged(["git", "fetch", "origin"], cwd=REPO)
run_logged(["git", "checkout", "--detach", REVISION], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "matplotlib>=3.9,<4"], cwd=REPO)
run_logged(["git", "rev-parse", "HEAD"], cwd=REPO)

In [ ]:
import torch

print(
    "Python:", sys.version, "PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, flush=True
)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training")
GPU = torch.cuda.get_device_name(0)
print(
    "GPU:", GPU, "VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30, flush=True
)
ALLOW_OTHER_GPU = False
if "A100" not in GPU and not ALLOW_OTHER_GPU:
    raise RuntimeError("Select A100, or explicitly set ALLOW_OTHER_GPU=True for another GPU")
PRECISION = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("Precision:", PRECISION, flush=True)

## Lưu trên Drive và kiểm tra input

Cell này yêu cầu quyền Drive trong Colab. Nếu chưa có bundle trên Drive, chọn đúng file ZIP đã chuẩn bị khi hộp upload xuất hiện. SHA256 được kiểm tra trước khi giải nén; không tải lại hay fit tokenizer.


In [ ]:
import tempfile

from google.colab import drive, files

from llm_scratch.data.training_bundle import extract_bundle

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/GRAST/wp07/pretraining")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = DRIVE_ROOT / "grast-wp07-data-block256-17971c0168e4.zip"
BUNDLE_SHA256 = "9f08cdc7ba7c8a4825ce90470fd19fc3ed6da7d6d3bfd742097a389d1aab949d"
if not BUNDLE.exists():
    uploaded = files.upload()
    if set(uploaded) != {BUNDLE.name}:
        raise ValueError("Upload exactly " + BUNDLE.name)
    BUNDLE.write_bytes(uploaded[BUNDLE.name])
INPUTS = extract_bundle(
    BUNDLE,
    Path(tempfile.mkdtemp(prefix="grast-inputs-")) / "verified",
    expected_sha256=BUNDLE_SHA256,
)
print("Verified inputs:", INPUTS, flush=True)

## Smoke trước, rồi train cùng recipe

`RESUME` là đường dẫn đầy đủ đến `checkpoints/latest.pt` đã lưu trên Drive. Để tiếp tục sau smoke hoặc sau mất runtime, giữ nguyên source commit, recipe, precision và runtime tương thích, điền `RESUME`, rồi chạy lại. Mỗi lần tạo run directory mới; lịch sử loss đi theo checkpoint và tự dựng lại biểu đồ.

`SMOKE_ONLY` chỉ đặt điểm dừng ở update 4, không đổi budget/lịch LR 2.000 updates. Recipe: model 46.148.096 tham số, block256, microbatch2, accumulation8, full held-out validation mỗi100 updates, latest checkpoint mỗi100 updates và các mốc100/500/1000/2000. Checkpoint đầy đủ khoảng0,52 GiB; các mốc được giữ lại trên Drive. Đọc các dòng `runner_checkpoint_started` và `checkpoint_saved` để biết tiến trình ghi.

Nếu runtime bị kill/OOM giữa update, tiếp tục từ latest đã hoàn tất; không có checkpoint cho gradients dở. Nếu GPU/Torch/recipe khác checkpoint, exact resume sẽ từ chối. Không tự bỏ optimizer/data state để ép chạy.


In [ ]:
SMOKE_ONLY = True
RESUME = ""  # e.g. /content/drive/MyDrive/GRAST/wp07/pretraining/runs/.../checkpoints/latest.pt
OUTPUT_ROOT = DRIVE_ROOT / "runs"
command = [
    sys.executable,
    "-u",
    "-m",
    "llm_scratch.training.pretrain",
    "--config",
    REPO / "configs/training/wp07_pretrain.toml",
    "--prepared-root",
    INPUTS / "pretraining_data",
    "--tokenizer",
    INPUTS / "tokenizer",
    "--device",
    "cuda",
    "--precision",
    PRECISION,
    "--output-root",
    OUTPUT_ROOT,
]
if SMOKE_ONLY:
    command += ["--stop-after", "4"]
if RESUME:
    if not Path(RESUME).is_file():
        raise FileNotFoundError(RESUME)
    command += ["--resume", RESUME]
run_logged(command, cwd=REPO)

In [ ]:
import json

from IPython.display import Image, display

runs = sorted(OUTPUT_ROOT.iterdir(), key=lambda p: p.name)
RUN = runs[-1]
print("Run:", RUN, flush=True)
print("Resume from:", RUN / "checkpoints/latest.pt", flush=True)
report_path = RUN / "artifacts/pretraining_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(
        json.dumps(
            {
                k: report[k]
                for k in (
                    "status",
                    "step",
                    "planned_steps",
                    "tokens_optimized",
                    "dataset_fingerprint",
                )
            },
            indent=2,
        ),
        flush=True,
    )
if (RUN / "artifacts/loss_curves.png").exists():
    display(Image(filename=str(RUN / "artifacts/loss_curves.png")))
print("Persistent files: config.json, metadata.json, logs/, checkpoints/, artifacts/", flush=True)

## Gửi kết quả về project

Cell cuối đóng gói report/curves/config/log (không đóng gói weights lớn) để chia sẻ. Giữ `latest.pt`, `final.pt` và các checkpoint mốc trên Drive cho WP07.4/5. Sau smoke 4 updates, chưa đánh dấu corpus baseline hoàn tất; cần chạy budget đã chọn và xem chất lượng riêng.


In [ ]:
import zipfile

SUMMARY = DRIVE_ROOT / (RUN.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.suffix not in {".pt", ".u32"}:
            archive.write(path, path.relative_to(RUN))
print("Summary:", SUMMARY, flush=True)
files.download(str(SUMMARY))